In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

# Crop Type Mapping
_Written by: Yi-Chia Chang_

In this tutorial, we will demonstrate how to train a model for a 3D semantic segmentation task using TorchGeo. We will train a [ConvLSTM](https://docs.torchgeo.org/en/stable/api/models/convlstm.html) semantic segmentation model using the satellite imagery and labels sampled from [Panoptic Segmentation of Satellite image Time Series (PASTIS)](https://torchgeo.readthedocs.io/en/stable/api/datasets/pastis.html) dataset to detect crop types in France using time-series Sentinel-2 imagery. By ingesting time-series Sentinel-2 imagery and crop-type labels, the model can predict pixel-level crop types.

It is recommended to use a GPU to accelerate the training and inference process. The demo was tested on Google Colab.

## Setup

In [ ]:
%pip install git+https://github.com/torchgeo/torchgeo.git

## Imports

In [ ]:
%matplotlib inline

import os
import tempfile

import lightning as L
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from torchgeo.datamodules import PASTIS100DataModule
from torchgeo.datasets import PASTIS100
from torchgeo.trainers import SpatioTemporalSegmentationTask

L.seed_everything(0, workers=True)

## Visualize the dataset

PASTIS100 is a subset of 100 patches sampled from the original [PASTIS](https://docs.torchgeo.org/en/stable/api/datasets/pastis.html) dataset. Each patch is 128×128 pixels with a variable-length Sentinel-2 time-series and a static 20-class semantic mask. We use all 10 L2A bands as training inputs. We split the dataset into 60 training, 20 validation, and 20 test samples. Here, we randomly visualize one sample in the dataset. The default visualization shows the first two images in the time-series and the corresponding label mask.

In [ ]:
import random

random.seed(50)

root = os.path.join(tempfile.gettempdir(), 'pastis100')
dataset = PASTIS100(root=root, bands=PASTIS100.s2_bands, download=True)

fig = dataset.plot(dataset[random.randrange(len(dataset))])
plt.show()

## Datamodule

We utilize TorchGeo's [Lightning](https://lightning.ai/docs/pytorch/stable/) datamodules to organize the dataloader setup. `PASTIS100DataModule` handles the train-val-test split, normalization, and the temporal padding needed for variable-length time series.

- Train-val-test split: The 100 patches are randomly split into 60 training, 20 validation, and 20 test samples. The split uses a fixed random seed, so it is reproducible across runs and machines.

- Normalization: Sentinel-2 L2A products store surface reflectance as integers scaled by 10,000. The datamodule divides each pixel by 10,000, converting the inputs to reflectance values in [0, 1]. The normalization is applied to every frame of the time series on the GPU after batch transfer.

- Data augmentation: No random augmentation is applied in this tutorial. The inputs are only normalized. If you want to add augmentation (e.g., random flips or crops), you can override datamodule.aug with additional Kornia transforms; wrapping them in K.VideoSequential ensures the same spatial transform is applied consistently to all timesteps and to the mask.

- Padding: Like many satellite image time series (SITS) datasets, PASTIS contains a variable number of time steps per location, ranging from 38 to 61. To keep tensor dimensions consistent within a batch, sequences shorter than `padding_length` are zero-padded at the end, and longer sequences are truncated to the first `padding_length` observations. In this tutorial we set `padding_length=38` to save memory during training, meaning some samples are truncated.

In [ ]:
batch_size = 4
num_workers = 4
max_epochs = 50
padding_length = 38
fast_dev_run = False

In [ ]:
datamodule = PASTIS100DataModule(
    root=root,
    bands=PASTIS100.s2_bands,
    batch_size=batch_size,
    num_workers=num_workers,
    padding_length=padding_length,
)

## Spatio-temporal Segmentation Task

`SpatioTemporalSegmentationTask` is a module designed for time series imagery with static segmentation labels. Convolutional Long Short-Term Memory ([ConvLSTM](https://docs.torchgeo.org/en/stable/api/models/convlstm.html)) has been shown to be effective for precipitation nowcasting ([Shi et al., 2015](https://arxiv.org/abs/1506.04214)). The convolutional structures are applied in the sequential transitions in ConvLSTM, enabling end-to-end training for crop type mapping.

- Parameters for the dataset: `in_channels` is set to 10 for using all bands in Sentinel-2 imagery. `num_classes=20` matches PASTIS's 18 crop categories plus background and a void class for parcels lying mostly outside their patch. Since void pixels carry no supervisory signal, we set `ignore_index=19` to exclude them from both the loss and the evaluation metrics.

- Parameters for the model: `hidden_dim=64` sets the number of channels in each ConvLSTM hidden state, `num_layers=2` stacks two ConvLSTM layers for a deeper temporal representation, and `kernel_size=3` gives the recurrent convolutions a 3×3 spatial neighborhood, the standard choice in convolutional networks. Larger values of `hidden_dim` and `num_layers` increase expressiveness at the cost of memory and a greater risk of overfitting since we train on only 60 patches. These moderate values work well for PASTIS100. Users training on the full PASTIS dataset can afford larger models.

- Parameters for training optimization: Cross-entropy (`loss='ce'`) is the standard objective for multi-class semantic segmentation. The task also supports `'jaccard'`, `'focal'`, and `'dice'` losses, which can help when classes are highly imbalanced. We lower the learning rate (`lr`) from the default 1e-3 to 1e-4, which we found trains more stably on this small dataset.

In [ ]:
model = SpatioTemporalSegmentationTask(
    model='convlstm',
    in_channels=10,
    num_classes=20,
    ignore_index=19,
    loss='ce',
    lr=1e-4,
    hidden_dim=64,
    num_layers=2,
    kernel_size=3,
)

## Training a deep learning model from scratch

We use Lightning's `Trainer` with TorchGeo's `SpatioTemporalSegmentationTask` to simplify training. The `EarlyStopping` callback terminates training when `val_loss` stops improving. The `patience` parameter specifies how many consecutive validation checks without improvement are tolerated before stopping, preventing overfitting and unnecessary computation once the model has plateaued. The `ModelCheckpoint` callback saves the checkpoint with the lowest `val_loss` during training. `.fit()` initiates the entire training and validation loop when organizing the PyTorch code into a `LightningModule` ([ref](https://lightning.ai/docs/pytorch/stable/common/trainer.html)). 

To achieve the best results, users may train their own models on the full PASTIS dataset.

In [ ]:
trainer = Trainer(
    accelerator='auto',
    callbacks=[
        EarlyStopping('val_loss', patience=5),
        ModelCheckpoint(monitor='val_loss', mode='min'),
    ],
    fast_dev_run=fast_dev_run,
    log_every_n_steps=1,
    max_epochs=max_epochs,
)
trainer.fit(model=model, datamodule=datamodule)

### Visualize training process

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

## Test the model
We evaluate the model on the test set using the checkpoint that achieved the highest performance on the validation set.

In [ ]:
ckpt_path = None if fast_dev_run else 'best'
trainer.test(model=model, datamodule=datamodule, ckpt_path=ckpt_path)

## Visualizing prediction results on test data

We can manually load and preprocess one test batch, predict segmentation masks, and visualize predictions against ground truth for 3 samples in the first test batch.

In [ ]:
datamodule.setup('test')
model.eval()

batch = datamodule.aug(next(iter(datamodule.test_dataloader())))

with torch.no_grad():
    predictions = model(batch['image']).argmax(dim=1)

for i in range(3):
    sample = {k: batch[k][i] for k in ('image', 'mask')}
    sample['prediction'] = predictions[i]
    dataset.plot(sample, suptitle=f'Test sample {i}')
    plt.show()